# MIL test inference

Attach the soil competition data and soil-cv-utils version 9.
Turn Internet on and allow WANDB_API_KEY for this notebook.
We use three ready MIL runs. Two slots are empty.
One ready checkpoint gives one CSV. Runs without a committed model are skipped.
There is no training, averaging between slots, or new W&B run.

In [ ]:
from pathlib import Path

CHECKPOINTS = [
    'public-team/soil-grain-size/uvi5xre4',
    'public-team/soil-grain-size/nn0qjyik',
    'public-team/soil-grain-size/uwkzhzxv',
    None,
    None,
]
DEVICE = 'auto'
OUTPUT_DIR = Path('/kaggle/working/submissions_mil')

## Load the MIL code

Use a run ID, a W&B/Forge run URL, or an exact model artifact version.
The notebook finds the model artifact logged by each run.
Only MIL models with the matching source hash can be loaded.

In [ ]:
import os
import sys
import torch
import wandb
from kaggle_secrets import UserSecretsClient

code_root = Path(os.environ.get('SOILCV_INFERENCE_CODE_ROOT','/kaggle/input/datasets/denysmykhailov/soil-cv-utils'))
if not (code_root/'inference/runner.py').is_file():
    matches = list(Path('/kaggle/input').rglob('inference/runner.py'))
    if len(matches) != 1:
        raise RuntimeError('Attach the latest soil-cv-utils dataset with inference/runner.py.')
    code_root = matches[0].parents[1]
sys.path.insert(0,str(code_root/'inference'))
from runner import run_checkpoints

try:
    key = UserSecretsClient().get_secret('WANDB_API_KEY')
except Exception as error:
    raise RuntimeError('Enable WANDB_API_KEY in Add-ons > Secrets for this notebook, then Run All.') from error
os.environ['WANDB_API_KEY'] = key
wandb.login(key=key,verify=True)
del key
print('Filled slots:',sum(isinstance(handle,str) and bool(handle.strip()) for handle in CHECKPOINTS))
print('GPU available:',torch.cuda.is_available())

## Predict the whole test set

Each model uses its own scale, grid, resolution and normalization config.
All photos and all grid tiles are used. Empty slots and unfinished checkpoints do not block ready models.
CSV rows follow sample_submission.csv. IDs and valid curves are checked.

In [ ]:
results = run_checkpoints(CHECKPOINTS,code_root,OUTPUT_DIR,device=DEVICE)
print('Saved CSV files:',len(results))

## Ready submissions

Each CSV belongs to one exact checkpoint. There is no ensemble.
inference_manifest.json records model versions, source runs and skipped references.

In [ ]:
import pandas as pd
from IPython.display import display,FileLink

assert results, 'No ready MIL predictions yet.'
for result in results:
    path = Path(result['output'])
    frame = pd.read_csv(path)
    print(path.name,'| shape:',frame.shape)
    display(FileLink(str(path)))
    display(frame.head(2))
display(pd.DataFrame(results)[['slot','artifact','device','output']])
print('Outputs:',OUTPUT_DIR)